# Steps to running offline twin

You will need to have ollama installed and any offline model installed to run this code.

You can use the default .venv kernel used in the course.

To send email through Gmail SMTP, put these in your `.env`:

```
GMAIL_USER=you@gmail.com
GMAIL_APP_PASSWORD=your-16-char-app-password
GMAIL_TO=you@gmail.com
```

Use a [Google App Password](https://myaccount.google.com/apppasswords), not your normal Gmail password. 2-Step Verification mustrtual be on.

create a virtual environment - 
uv venv

install the packages - 
uv pip install -r requirements.txt


In [9]:
# imports

from dotenv import load_dotenv
from openai import OpenAI
import json
import os
import smtplib
import requests
from email.message import EmailMessage
from pypdf import PdfReader
import gradio as gr

In [10]:
load_dotenv(override=True)
openai = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

In [11]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

In [12]:
gmail_user = os.getenv("GMAIL_USER")
gmail_app_password = (os.getenv("GMAIL_APP_PASSWORD") or "").replace(" ", "") or None
gmail_to = os.getenv("GMAIL_TO")

In [13]:
def send_email(subject, body, to=None):
    recipient = to or gmail_to
    if not gmail_user or not gmail_app_password or not recipient:
        print("Email skipped: set GMAIL_USER, GMAIL_APP_PASSWORD, and GMAIL_TO in .env")
        return "Email not sent: missing Gmail credentials"

    message = EmailMessage()
    message["From"] = gmail_user
    message["To"] = recipient
    message["Subject"] = subject
    message.set_content(body)

    try:
        with smtplib.SMTP_SSL("smtp.gmail.com", 465, timeout=30) as smtp:
            smtp.login(gmail_user, gmail_app_password)
            smtp.send_message(message)
    except Exception as e:
        print(f"Email failed: {e}")
        return f"Email not sent: {e}"

    print(f"Email sent to {recipient}: {subject}")
    return "OK"

In [14]:
def record_user_details(email, name="Name not provided", notes="not provided"):
    body = f"Name: {name}\nEmail: {email}\nNotes: {notes}"
    return send_email("Website visitor wants to get in touch", body, email)

In [15]:
def record_unknown_question(question):
    return send_email("Unknown question from website visitor", f"Question: {question}")

In [16]:
record_user_details_json = {
    "name": "record_user_details",
    "description": "Use this tool to record that a user is interested in being in touch and provided an email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {"type": "string", "description": "The email address of this user"},
            "name": {"type": "string", "description": "The user's name, if they provided it"},
            "notes": {"type": "string", "description": "Any additional info about the conversation that's worth recording to give context"
            }
        },
        "required": ["email"],
        "additionalProperties": False
    }
}

In [17]:
record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {"type": "string", "description": "The question that couldn't be answered"},
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

In [18]:
tools = [{"type": "function", "function": record_user_details_json},
        {"type": "function", "function": record_unknown_question_json}]

In [19]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [20]:
reader = PdfReader("./Profile.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

with open("./summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [21]:
system_prompt = f"""

# Your role

You are a digital twin running on a website, chatting with visitors of the website.
You represent the person who's website you are on.
You answer questions related to their career, background, skills and experience.

Here are the details of the person you are representing:

{summary}

If asked, you explain clearly that you are an AI that is the digital twin of this person.

# Context

Here is a summary of the person's LinkedIn profile so that you can answer questions:

{linkedin}

# Rules

Engage with the user. Be professional and engaging, as if talking to a potential client or future employer who came across the website.
Only answer questions related to career, background, skills and experience.
If the user asks about something unrelated, then steer the conversation back to professional topics.

Always stay in character as the digital twin of the person you are representing. Represent the person.

If the user would like to get in touch, then ask for their email, and use your tool to record their email for follow-up.

IMPORTANT:
If you don't know the answer, use your tool to record the question, and then tell the user that you don't know. Never make up an answer.
"""


In [22]:
models = requests.get('http://localhost:11434/v1/models').json()
print(models)
for model in models.get("data"):
    print(model.get("id"))

{'object': 'list', 'data': [{'id': 'cogito:14b', 'object': 'model', 'created': 1791391971, 'owned_by': 'library'}, {'id': 'gemma4:latest', 'object': 'model', 'created': 1791043445, 'owned_by': 'library'}, {'id': 'gemma3:latest', 'object': 'model', 'created': 1791034030, 'owned_by': 'library'}]}
cogito:14b
gemma4:latest
gemma3:latest


In [25]:
model = "cogito:14b"

def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=model, messages=messages, tools=tools)
    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        tool_calls = message.tool_calls
        results = handle_tool_calls(tool_calls)
        messages.append(message)
        messages.extend(results)
        response = openai.chat.completions.create(model=model, messages=messages, tools=tools)
    return response.choices[0].message.content

In [26]:
gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Email sent to sonicdas@gmail.com: Website visitor wants to get in touch
